# UA-FCCL final experiments (Colab)
Run the cells **one by one, from top to bottom**. Wait for each cell to finish before the next one. If this Colab session was used before, first click **Runtime > Restart session**.

**Before you start, have these two files on your computer:**
1. `ua_fccl.py` (the newest one I sent; delete any older copies with `(1)` in the name)
2. `Shill Bidding Dataset.csv` (the file you already used for the paper)

Total run time is about 1.5 to 2 hours. The three long cells (Cells 8, 9, 10) save their results after every dataset, so if Colab disconnects you only repeat the unfinished dataset.

## Cell 1: get your repo, the authors' clustered-conformal code, and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl class-conditional-conformal
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
!git clone -q https://github.com/tiffanyding/class-conditional-conformal.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
!ls ccl_helpers.py /content/class-conditional-conformal/utils

## Cell 2: download the datasets (ignore the Shill Bidding message)

In [ ]:
!python 01_prepare_data.py
!ls data

## Cell 3: upload `Shill Bidding Dataset.csv`
Click **Choose Files**, select the file, and wait. The cell prints nothing until you choose it.

In [ ]:
import os, shutil
from google.colab import files
up = files.upload()
for fn in up:
    shutil.move(fn, 'data/Shill Bidding Dataset.csv')
!ls data

## Cell 4: upload `ua_fccl.py`
Click **Choose Files** and select the newest `ua_fccl.py`. The second part of the cell fixes the file name if your computer added `(1)`.

In [ ]:
import glob, os, shutil
from google.colab import files
up = files.upload()
cands = sorted(glob.glob('ua_fccl*.py'), key=os.path.getmtime)
shutil.copy(cands[-1], 'ua_fccl.py')
print('using', cands[-1])
!ls ua_fccl.py

## Cell 5: download three more datasets with tiny classes
Classes with fewer than 8 cases are dropped. The cell prints what was dropped.

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
for name, i in {'shuttle': 148, 'yeast': 110, 'ecoli': 39}.items():
    d = fetch_ucirepo(id=i)
    Xf = d.data.features.select_dtypes('number')
    yt = d.data.targets.iloc[:, 0].astype(str)
    out = pd.concat([Xf, yt.rename('class')], axis=1)
    cnt = out['class'].value_counts()
    drop = cnt[cnt < 8].index.tolist()
    out = out[~out['class'].isin(drop)]
    out.to_csv(f'data/{name}.csv', index=False)
    print(name, out.shape, 'dropped:', drop, '| class sizes:', out['class'].value_counts().tolist())

## Cell 6: load the nine datasets and check that the clustered-conformal code works

In [ ]:
import numpy as np, pandas as pd, time, os
import ccl_helpers as H
import ua_fccl as U

files_map = {'shill': 'Shill Bidding Dataset.csv', 'wilt': 'wilt.csv', 'htru2': 'htru2.csv', 'magic': 'magic.csv',
             'pageblocks': 'pageblocks.csv', 'drybean': 'drybean.csv', 'shuttle': 'shuttle.csv',
             'yeast': 'yeast.csv', 'ecoli': 'ecoli.csv'}
data = {n: H.load('data/' + f) for n, f in files_map.items()}
for n, (X, y) in data.items():
    print(n, X.shape, 'class sizes:', np.bincount(y).tolist())

# quick test of the authors' code on random numbers
rng = np.random.RandomState(0)
test_scores = rng.rand(300, 3); test_y = rng.randint(0, 3, 300)
print('clustered code OK:', U.sets_clustered(test_scores, test_y, test_scores[:5], 0.1).shape)

## Cell 7: a helper that runs datasets and saves results after each one

In [ ]:
seeds_for = {'pageblocks': range(50)}

def run_group(names):
    for n in names:
        f = f'final_splits_{n}.csv'
        if os.path.exists(f):
            print(n, 'already done, skipping'); continue
        t0 = time.time()
        X, y = data[n]
        s, c, g, fails = U.run_all(n, X, y, seeds_for.get(n, range(20)))
        s.to_csv(f, index=False)
        c.to_csv(f'final_classes_{n}.csv', index=False)
        g.to_csv(f'final_gate_{n}.csv', index=False)
        print(n, 'done in', round((time.time() - t0) / 60, 1), 'min | methods that failed to run:', fails or 'none')

## Cell 8: small datasets (about 15 minutes)

In [ ]:
run_group(['ecoli', 'yeast', 'wilt', 'pageblocks'])

## Cell 9: medium datasets (about 30 to 40 minutes)

In [ ]:
run_group(['shill', 'drybean', 'htru2', 'magic'])

## Cell 10: Shuttle, the largest (about 20 to 30 minutes)

In [ ]:
run_group(['shuttle'])

## Cell 11: combine and summarise
This prints two tables.
- **Table A** (rare classes): `rare_cov` is the share of the rarest class covered (pooled over all splits), `worst_class_cov` is the worst class, `answered` is the share of cases with exactly one label. `target` is `natural` (all calibration cases) or the number of rare-class calibration cases we kept.
- **Table B** (gate): `share` is the share of cases with an explanation shown, `unfaith_among_shown` should be at or below `delta`.

In [ ]:
names = list(files_map)
splits = pd.concat([pd.read_csv(f'final_splits_{n}.csv') for n in names])
classes = pd.concat([pd.read_csv(f'final_classes_{n}.csv') for n in names])
gate = pd.concat([pd.read_csv(f'final_gate_{n}.csv') for n in names])
TA = U.summarise_module1(splits, classes)
TB = U.summarise_gate(gate)
TA.to_csv('final_tableA_rare_classes.csv')
TB.to_csv('final_tableB_gate.csv')
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 2000)
print('TABLE A: forest score, smallest calibration size')
for n in names:
    d = TA.loc[n]
    d = d[d.index.get_level_values('score') == 'forest']
    tg = [t for t in ['4', '8', 'natural'] if t in d.index.get_level_values('target')]
    print('\n==', n, '==')
    print(d.loc[tg[0]].to_string())

## Cell 12: Table B, the gate

In [ ]:
print(TB.to_string())

## Cell 13: download everything and send it to me

In [ ]:
!zip -q final_results.zip final_*.csv
from google.colab import files
files.download('final_results.zip')